In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import torch
import gc
from sklearn.metrics import accuracy_score, average_precision_score
from sklearn.preprocessing import LabelEncoder, label_binarize
from aeon.datasets import load_classification
from aeon.datasets.tsc_datasets import univariate as ucr_names
from tqdm.notebook import tqdm
from datetime import datetime

# Add parent directory to path for imports
%cd ..
try:
    from distances import TimeSeriesDistance
    print("Successfully imported TimeSeriesDistance from distances.py")
except ImportError as e:
    print(f"Could not import distances.py: {e}")
%cd test_knn
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# --- PATHS ---
DATA_PATH = "../consolidated_datasets/aeon_datasets"
RESULTS_FILE = "multi_distance_knn_results.csv"
ERROR_LOG_FILE = "multi_distance_failed_log.txt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- DATASET SELECTION (UCR subset) ---
SELECTED_DATASETS = [
    "BME",
    "CBF",
    "Chinatown",
    "DodgerLoopDay",
    "DodgerLoopWeekend",
    "FreezerRegularTrain",
    "FreezerSmallTrain",
    "GesturePebbleZ1",
    "GesturePebbleZ2",
    "GunPointAgeSpan",
    "GunPointMaleVersusFemale",
    "Lightning7",
    "MelbournePedestrian",
    "PickupGestureWiimoteZ",
    "ShakeGestureWiimoteZ",
    "SmoothSubspace",
    "ToeSegmentation1",
    "ToeSegmentation2",
    "Trace",
    "UMD",
]

# --- KNN CONFIG ---
# Multiple k values to test
KNN_K_VALUES = [1, 3, 5, 7, 9, 11, 13, 15]

# --- DISTANCES TO RUN ---
# Comment/uncomment distances you want to test
DISTANCES_TO_RUN = [
    "tmw",
    "opw",
    "taot", 
    "tcot",
    "awswd",
    "otw",      
    "dtw",
    "gow",
    "pow"
]

# --- PER-DISTANCE HYPERPARAMETERS ---
# These override the defaults in the TimeSeriesDistance wrapper
DISTANCE_HYPERPARAMS = {
    "opw": {
        "lambda1": 50.0,
        "lambda2": 0.1,
        "sigma": 1.0,
        "num_iter": 100
    },
    "taot": {
        "reg_lambda": 10.0,
        "time_weight": 10.0,
        "num_iter": 1000
    },
    "tcot": {
        "reg_lambda": 10.0,
        "num_iter": 1000
    },
    "awswd": {
        "reg_lambda": 10.0,
        "l_window": 5,
        "k_steep": 0.1,
        "num_sinkhorn": 100,
        "num_outer": 5
    },
    "tmw": {
        "cost_function": "L2",
        "mask_type": 2,
        "reg": 0.01,
        "max_iterations": 2000,
        "thres": 1e-5,
        "eps_threshold": 0.2,
        "masked": True,
        "rescale": True,
    },
    "otw": {
        "m_cost": 1.0,
        "s_window": -1,  # -1 means global (s=n)
        "beta_smooth_l1": 1.0,
        "strategy_neg": "direct"  # "direct" or "split_pos_neg"
    },
    "dtw": {
        "global_constraint": None,
        "sakoe_chiba_radius": None,
        "itakura_max_slope": None
    },
    'gow': {
        'lambda1': 5.0,
        'lambda2': 10.0,
        'max_iter': 15,
        'sinkhorn_iter': 200,
        'fw_iter': 100
    },
    'pow': {
        'order_reg': 1.0,
        'sinkhorn_reg': 0.1,
        'm_mass': 0.8,
        'num_iter': 200
    },
    "euclidean": {}
}

print(f"Device: {DEVICE}")
print(f"Distances to run: {DISTANCES_TO_RUN}")
print(f"KNN k values: {KNN_K_VALUES}")
print(f"Selected datasets: {len(SELECTED_DATASETS)}")

In [ ]:
# ============================================================================
# CHECKPOINT MANAGER
# ============================================================================

def get_processed_entries():
    """
    Returns a set of (dataset, distance) tuples that have already been processed.
    """
    if not os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, 'w') as f:
            f.write("dataset,distance,k,accuracy,mAP,length,n_train,n_test,n_distances,time_sec\n")
        return set()
    
    try:
        df = pd.read_csv(RESULTS_FILE)
        if 'dataset' not in df.columns or 'distance' not in df.columns:
            return set()
        return set(zip(df['dataset'], df['distance']))
    except Exception:
        return set()


def save_result(dataset, distance_name, k, acc, map_score, length, n_train, n_test, duration):
    """Save best-k result to the CSV file."""
    n_dists = n_train * n_test
    with open(RESULTS_FILE, 'a') as f:
        f.write(f"{dataset},{distance_name},{k},{acc:.5f},{map_score:.5f},{length},{n_train},{n_test},{n_dists},{duration:.2f}\n")


def log_failure(dataset, distance_name, error_msg):
    """Log a failure to the error log file."""
    with open(ERROR_LOG_FILE, 'a') as f:
        f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {dataset} ({distance_name}): {error_msg}\n")

In [ ]:
# ============================================================================
# BATCHING UTILITIES
# ============================================================================

def get_optimal_batch_size(series_length, vram_fraction=0.8):
    """
    Estimate optimal batch size based on available VRAM and series length.
    """
    if str(DEVICE) == 'cpu':
        return 512
    try:
        free_mem, _ = torch.cuda.mem_get_info()
        usable_mem = free_mem * vram_fraction
        
        # Estimate: (L*L*4 bytes cost matrix) * 4 overhead + inputs
        est_mem = (series_length ** 2 * 4) * 4 + (series_length * 8)
        
        batch_size = int(usable_mem / est_mem)
        return max(1, min(batch_size, 4096))
    except:
        return 64

In [ ]:
# ============================================================================
# KNN ENGINE
# ============================================================================

def compute_distance_matrix(X_train, X_test, distance_func):
    """
    Precompute full distance matrix between test and train samples.
    
    Args:
        X_train: numpy array of shape (n_train, seq_len) or (n_train, seq_len, dims)
        X_test: numpy array of shape (n_test, seq_len) or (n_test, seq_len, dims)
        distance_func: TimeSeriesDistance instance
        
    Returns:
        numpy array of shape (n_test, n_train) with pairwise distances
    """
    n_train = X_train.shape[0]
    n_test = X_test.shape[0]
    length = X_train.shape[1]
    
    # Add feature dimension if needed (shape: n, seq_len, 1)
    if X_train.ndim == 2:
        X_train = X_train[:, :, np.newaxis]
        X_test = X_test[:, :, np.newaxis]
    
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    
    batch_size = get_optimal_batch_size(length)
    
    # Initialize distance matrix
    dist_matrix = np.zeros((n_test, n_train), dtype=np.float32)
    
    for i in range(n_test):
        test_seq = X_test_t[i].unsqueeze(0)  # Shape: (1, seq_len, dims)
        
        for j in range(0, n_train, batch_size):
            end_j = min(j + batch_size, n_train)
            current_bs = end_j - j
            
            batch_train = X_train_t[j:end_j]  # Shape: (bs, seq_len, dims)
            batch_test = test_seq.repeat(current_bs, 1, 1)  # Shape: (bs, seq_len, dims)
            
            dists = distance_func(batch_test, batch_train)  # Shape: (bs,)
            dist_matrix[i, j:end_j] = dists.cpu().numpy()
        
        if (i + 1) % 100 == 0:
            print(f"    Computing distances: {i + 1}/{n_test}...", end='\r')
    
    return dist_matrix


def run_knn_from_distance_matrix(dist_matrix, y_train, y_test, k_values):
    """
    Run k-NN classification for multiple k values using precomputed distance matrix.
    
    Args:
        dist_matrix: numpy array of shape (n_test, n_train) with pairwise distances
        y_train: numpy array of training labels
        y_test: numpy array of test labels
        k_values: list of k values to test
        
    Returns:
        dict: {k: {'accuracy': acc, 'mAP': map_score}} for each k value
    """
    from collections import Counter
    
    n_test = dist_matrix.shape[0]
    n_train = dist_matrix.shape[1]
    max_k = max(k_values)
    
    # Handle case where max_k > n_train
    effective_max_k = min(max_k, n_train)
    
    # Get indices of top max_k nearest neighbors for each test sample
    top_k_indices = np.argpartition(dist_matrix, effective_max_k - 1, axis=1)[:, :effective_max_k]
    
    # Sort these top-k indices by actual distance
    row_indices = np.arange(n_test)[:, np.newaxis]
    top_k_dists = dist_matrix[row_indices, top_k_indices]
    sorted_order = np.argsort(top_k_dists, axis=1)
    top_k_indices_sorted = np.take_along_axis(top_k_indices, sorted_order, axis=1)
    
    # Determine all unique classes
    all_classes = np.unique(np.concatenate([y_train, y_test]))
    n_classes = len(all_classes)
    
    # Binarize true labels for mAP calculation (One-vs-Rest)
    y_test_bin = label_binarize(y_test, classes=all_classes)
    if n_classes == 2:
        y_test_bin = np.hstack([1 - y_test_bin, y_test_bin])
    
    results = {}
    
    for k in k_values:
        if k > n_train:
            print(f"    Warning: k={k} > n_train={n_train}, skipping.")
            continue
            
        predictions = []
        y_score = np.zeros((n_test, n_classes), dtype=np.float64)
        
        for i in range(n_test):
            k_nearest_indices = top_k_indices_sorted[i, :k]
            k_nearest_labels = y_train[k_nearest_indices]
            
            # Majority vote
            label_counts = Counter(k_nearest_labels)
            most_common = label_counts.most_common(1)[0][0]
            predictions.append(most_common)
            
            # KNN probabilities
            for cls_idx, cls in enumerate(all_classes):
                y_score[i, cls_idx] = label_counts.get(cls, 0) / k
        
        acc = accuracy_score(y_test, predictions)
        
        # Compute mAP
        try:
            ap_per_class = []
            for cls_idx in range(n_classes):
                if y_test_bin[:, cls_idx].sum() > 0:
                    ap = average_precision_score(y_test_bin[:, cls_idx], y_score[:, cls_idx])
                    ap_per_class.append(ap)
            map_score = np.mean(ap_per_class) if ap_per_class else 0.0
        except Exception:
            map_score = 0.0
        
        results[k] = {'accuracy': acc, 'mAP': map_score}
    
    return results


def run_knn_with_distance(X_train, y_train, X_test, y_test, distance_func, k=1):
    """
    Run k-NN classification using a custom distance function (legacy single-k version).
    
    Args:
        X_train: numpy array of shape (n_train, seq_len)
        y_train: numpy array of labels
        X_test: numpy array of shape (n_test, seq_len)
        y_test: numpy array of labels
        distance_func: TimeSeriesDistance instance
        k: number of neighbors (default 1)
        
    Returns:
        dict: {'accuracy': acc, 'mAP': map_score}
    """
    dist_matrix = compute_distance_matrix(X_train, X_test, distance_func)
    results = run_knn_from_distance_matrix(dist_matrix, y_train, y_test, [k])
    return results[k]

In [ ]:
# ============================================================================
# MAIN EXECUTION - MULTI-DISTANCE TESTING
# ============================================================================

def run_multi_distance_knn():
    """
    Main function to run KNN with multiple distance functions and multiple k values on UCR datasets.
    """
    # Get already processed entries
    processed_entries = get_processed_entries()
    
    # Keep dataset order as defined in SELECTED_DATASETS
    all_datasets = list(SELECTED_DATASETS)
    
    # Find datasets that need processing for any distance
    pending_list = []
    for dataset in all_datasets:
        needs_processing = False
        for dist_name in DISTANCES_TO_RUN:
            if (dataset, dist_name) not in processed_entries:
                needs_processing = True
                break
        if needs_processing:
            pending_list.append(dataset)
    
    if not pending_list:
        print("All datasets completed for all distances!")
        return
    
    print(f"Found {len(pending_list)} datasets needing processing.")
    print(f"Testing k values: {KNN_K_VALUES}")
    print(f"\n[Phase 2] Starting Processing ({len(pending_list)} Datasets x {len(DISTANCES_TO_RUN)} Distances)")
    print("=" * 80)
    
    # Initialize distance functions
    distance_funcs = {}
    for dist_name in DISTANCES_TO_RUN:
        params = DISTANCE_HYPERPARAMS.get(dist_name, {})
        try:
            distance_funcs[dist_name] = TimeSeriesDistance(dist_name, params, device=DEVICE)
            print(f"  Initialized {dist_name.upper()} distance")
        except Exception as e:
            print(f"  [!] Failed to initialize {dist_name}: {e}")
    
    print("=" * 80)
    
    # Process each dataset
    for i, dataset_name in enumerate(pending_list):
        print(f"\n[{i + 1}/{len(pending_list)}] {dataset_name}")
        
        try:
            # Load data once per dataset
            X_train, y_train_raw = load_classification(dataset_name, split="train", extract_path=DATA_PATH)
            X_test, y_test_raw = load_classification(dataset_name, split="test", extract_path=DATA_PATH)
            
            if X_train.ndim == 3:
                X_train = X_train.squeeze(1)
                X_test = X_test.squeeze(1)
            
            # Handle NaNs
            X_train = np.nan_to_num(X_train)
            X_test = np.nan_to_num(X_test)
            
            # Basic dataset stats
            length = X_train.shape[1]
            n_train = X_train.shape[0]
            n_test = X_test.shape[0]
            complexity = n_train * n_test * (length ** 2)
            print(f"    L={length}, Train={n_train}, Test={n_test}, Complexity={complexity:,}")
            
            # Label encoding
            le = LabelEncoder()
            all_labels = np.concatenate([y_train_raw, y_test_raw])
            le.fit(all_labels)
            y_train = le.transform(y_train_raw)
            y_test = le.transform(y_test_raw)
            
            # Test each distance function
            for dist_name, dist_func in distance_funcs.items():
                if (dataset_name, dist_name) in processed_entries:
                    print(f"    [{dist_name.upper()}] Already processed, skipping.")
                    continue
                
                print(f"    [{dist_name.upper()}] Computing distances and selecting best k...")
                
                try:
                    start_ts = time.time()
                    
                    # Compute distance matrix once
                    dist_matrix = compute_distance_matrix(X_train, X_test, dist_func)
                    dist_compute_time = time.time() - start_ts
                    
                    # Run KNN for all k values using precomputed distances
                    knn_start = time.time()
                    k_results = run_knn_from_distance_matrix(dist_matrix, y_train, y_test, KNN_K_VALUES)
                    knn_time = time.time() - knn_start
                    
                    if not k_results:
                        print("    No valid k values for this dataset.")
                        log_failure(dataset_name, dist_name, "No valid k values")
                        continue
                    
                    total_time = time.time() - start_ts
                    
                    # Select the best k by accuracy, then mAP
                    best_k = max(k_results, key=lambda k: (k_results[k]['accuracy'], k_results[k]['mAP']))
                    best_acc = k_results[best_k]['accuracy']
                    best_map = k_results[best_k]['mAP']
                    
                    print(f"    Best: k={best_k} -> Acc={best_acc:.4f}, mAP={best_map:.4f} | Total time: {total_time:.2f}s")
                    save_result(dataset_name, dist_name, best_k, best_acc, best_map,
                               length, n_train, n_test, dist_compute_time + knn_time)
                    
                except Exception as e:
                    print(f"    FAILED: {e}")
                    log_failure(dataset_name, dist_name, str(e))
                
                # Cleanup GPU memory after each distance
                torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"    !!! Dataset load FAILED: {e}")
            for dist_name in DISTANCES_TO_RUN:
                log_failure(dataset_name, dist_name, f"Dataset Load Error: {e}")
        
        # Cleanup
        torch.cuda.empty_cache()
        gc.collect()
    
    print("\n" + "=" * 80)
    print("Processing complete!")


# Run the main function
if __name__ == "__main__":
    run_multi_distance_knn()

In [ ]:
# ============================================================================
# RESULTS ANALYSIS
# ============================================================================

def analyze_results():
    """Analyze and display results from the multi-distance KNN experiments."""
    if not os.path.exists(RESULTS_FILE):
        print("No results file found.")
        return
    
    df = pd.read_csv(RESULTS_FILE)
    
    # Filter out skipped entries (accuracy = -1)
    df_valid = df[df['accuracy'] >= 0]
    
    # Add mAP column if missing (backward compatibility)
    if 'mAP' not in df_valid.columns:
        df_valid['mAP'] = 0.0
    
    print("\n" + "=" * 80)
    print("RESULTS SUMMARY")
    print("=" * 80)
    
    # Summary by distance (best k per dataset)
    print("\n--- Summary by Distance ---")
    summary = df_valid.groupby('distance').agg({
        'accuracy': ['mean', 'std', 'min', 'max', 'count'],
        'mAP': ['mean', 'std', 'min', 'max'],
        'k': lambda x: x.value_counts().to_dict(),
        'time_sec': ['mean', 'sum']
    }).round(4)
    print(summary)
    
    # Pivot table: datasets vs distances (accuracy)
    print("\n--- Accuracy by Dataset and Distance ---")
    pivot_acc = df_valid.pivot_table(
        index='dataset', 
        columns='distance', 
        values='accuracy',
        aggfunc='first'
    )
    print(pivot_acc.head(20))
    
    # Pivot table: mAP
    print("\n--- mAP by Dataset and Distance ---")
    pivot_map = df_valid.pivot_table(
        index='dataset', 
        columns='distance', 
        values='mAP',
        aggfunc='first'
    )
    print(pivot_map.head(20))
    
    # Best distance per dataset
    print("\n--- Best Distance per Dataset (Top 10) ---")
    best_per_dataset = df_valid.loc[df_valid.groupby('dataset')['accuracy'].idxmax()]
    print(best_per_dataset[['dataset', 'distance', 'k', 'accuracy', 'mAP']].head(10))
    
    # Count wins per distance
    print("\n--- Win Count by Distance ---")
    win_counts = best_per_dataset['distance'].value_counts()
    print(win_counts)
    
    # mAP summary
    print("\n--- Mean mAP by Distance ---")
    map_summary = df_valid.groupby('distance')['mAP'].agg(['mean', 'std']).round(4)
    print(map_summary)
    
    # Best k distribution
    print("\n--- Best k Distribution (across all dataset/distance pairs) ---")
    best_k_counts = df_valid['k'].value_counts().sort_index()
    print(best_k_counts)
    
    return df_valid, best_per_dataset

# Run analysis
results_df = analyze_results()